# Genetic algorithm baseline for the scheduling MILP

This notebook builds a first genetic algorithm (GA) baseline for the current thesis formulation. The goal is not to replace MILP; it is to create a second solver family that we can later compare against MILP, quantum/QC approaches, and hybrid methods.

The first version deliberately solves the **battery-free** scheduling problem. This keeps the GA focused on the discrete assignment question: which compatible cluster and start time should each job use? Battery dispatch is a continuous operational layer and can be added later as either a second optimization step or a decoder inside the GA.


## Why these instances?

For the first implementation we use the repo-local synthetic instances because they already match the current model schema and cluster assumptions. The Alibaba traces remain valuable, but they still need a workload-to-resource mapping layer before they are safe for solver benchmarking.

The notebook defaults to `jobs_limit.csv` with a small job subset. This is intentional:

- `jobs_light.csv` is useful for smoke tests, but it is usually too easy.
- `jobs_tense.csv` is useful once the algorithm is stable.
- `jobs_limit.csv` is the best first stress test because it has narrower feasible windows and more pressure on shared resources.
- A subset lets us compare GA against MILP quickly and inspect feasibility manually. Once this works, increase `MAX_JOBS` or set it to `None`.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from time import perf_counter
from typing import Any

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.scenarios import build_repo_scenario
from src.evaluation.metrics import compute_summary_metrics
from src.evaluation.results import extract_hourly_results, extract_schedule
from src.milp.expressions import is_active
from src.milp.gurobi_model import build_milp_model
from src.milp.solve import solve_model

pd.set_option("display.max_columns", 120)
PROJECT_ROOT

PosixPath('/Users/juanparrado/Documents/Quantum Master/master_thesis/data_center_scheduling')

In [2]:
# Experiment controls.
# Increase MAX_JOBS after the notebook produces feasible schedules on the default setup.
WORKLOAD_CASE = "limit"
ENERGY_SCENARIO = "base"
MAX_JOBS: int | None = 15
PUE = 1.0
USE_BATTERY = False
MILP_TIME_LIMIT_SECONDS = 30

GA_POPULATION_SIZE = 60
GA_GENERATIONS = 80
GA_MUTATION_RATE = 0.08
GA_TOURNAMENT_SIZE = 4
GA_ELITE_COUNT = 4
GA_REPAIR_PASSES = 8
GA_PENALTY_WEIGHT = 1_000_000.0
GA_RANDOM_SEED = 7

jobs_df, hourly_df, clusters_df, config = build_repo_scenario(
    project_root=PROJECT_ROOT,
    workload_case=WORKLOAD_CASE,
    energy_scenario=ENERGY_SCENARIO,
    pue=PUE,
    use_battery=USE_BATTERY,
)

if MAX_JOBS is not None:
    jobs_df = jobs_df.head(MAX_JOBS).copy()

print(f"Jobs: {len(jobs_df)}")
print(f"Hours: {len(hourly_df)}")
print(f"Clusters: {len(clusters_df)}")
print(f"Battery enabled in config: {config.battery_power_capacity > 0 and config.battery_energy_capacity > 0}")

jobs_df.head()

Jobs: 15
Hours: 24
Clusters: 7
Battery enabled in config: False


,job_id,category,workload_family,tier,gpus,gpu_count_required,gpu_type_required,cpu_required,memory_required_gb,duration,duration_h,power_kw,e_kw,power,earliest_start,latest_start,alpha_B,alpha_C,alpha_D
0,FT-01,fine_tuning,fine_tuning,finetune,4,4,,32.0,192.0,4,4,3.54,3.54,0.00354,8,9,1,1,1
1,FT-02,fine_tuning,fine_tuning,finetune,4,4,,32.0,192.0,4,4,3.59,3.59,0.00359,9,10,1,1,1
2,FT-03,fine_tuning,fine_tuning,finetune,4,4,,32.0,192.0,4,4,3.53,3.53,0.00353,9,10,1,1,1
3,FT-04,fine_tuning,fine_tuning,finetune,4,4,,32.0,192.0,4,4,3.54,3.54,0.00354,9,10,1,1,1
4,FT-05,fine_tuning,fine_tuning,finetune,4,4,,32.0,192.0,4,4,3.58,3.58,0.00358,11,12,1,1,1


## Chromosome design and feasibility strategy

Each chromosome is a vector with one gene per job. A gene does **not** store arbitrary values. It stores an index into that job's precomputed feasible options:

`gene[j] -> (compatible_cluster, feasible_start_hour)`

This design guarantees two constraints by construction:

- every job is scheduled exactly once;
- every selected assignment respects the job's feasible start window and individual job-to-cluster compatibility.

The remaining hard constraints are shared per-cluster, per-hour capacities: MW, GPU count, CPU, and memory. Those cannot be guaranteed independently by one gene, because they depend on overlaps between many jobs. The GA therefore uses three safeguards:

1. **Repair:** after creating or mutating a chromosome, try to move jobs away from overloaded cluster-hours.
2. **Penalty:** infeasible chromosomes remain searchable, but receive a very large objective penalty.
3. **Final checker:** a schedule is only reported as solved if the checker finds zero capacity violations.


In [3]:
@dataclass(frozen=True)
class GAConfig:
    population_size: int = 80
    generations: int = 120
    mutation_rate: float = 0.08
    tournament_size: int = 4
    elite_count: int = 4
    repair_passes: int = 30
    penalty_weight: float = 1_000_000.0
    random_seed: int = 7


def prepare_ga_inputs(jobs_df: pd.DataFrame, hourly_df: pd.DataFrame, clusters_df: pd.DataFrame, config):
    """Use the MILP builder as the canonical source for compatibility and feasible starts."""
    model, variables = build_milp_model(jobs_df, hourly_df, clusters_df, config)
    model.Params.OutputFlag = 0

    job_ids = list(variables["jobs"].keys())
    options_by_job = {
        job_id: [
            (cluster, start)
            for cluster in variables["compatible_clusters"][job_id]
            for start in variables["feasible_starts"][job_id]
        ]
        for job_id in job_ids
    }
    empty_options = [job_id for job_id, options in options_by_job.items() if not options]
    if empty_options:
        raise ValueError(f"jobs without GA assignment options: {empty_options}")

    return model, variables, job_ids, options_by_job


milp_model_for_options, variables, job_ids, options_by_job = prepare_ga_inputs(jobs_df, hourly_df, clusters_df, config)
option_counts = pd.Series({job_id: len(options) for job_id, options in options_by_job.items()}, name="option_count")
option_counts.describe(), option_counts.head()

Set parameter Username


Set parameter LicenseID to value 2828833


Academic license - for non-commercial use only - expires 2027-05-29


(count    15.0
 mean     12.0
 std       0.0
 min      12.0
 25%      12.0
 50%      12.0
 75%      12.0
 max      12.0
 Name: option_count, dtype: float64,
 FT-01    12
 FT-02    12
 FT-03    12
 FT-04    12
 FT-05    12
 Name: option_count, dtype: int64)

In [4]:
def decode_individual(individual, job_ids, options_by_job):
    return {
        job_id: options_by_job[job_id][int(gene)]
        for job_id, gene in zip(job_ids, individual, strict=True)
    }


def build_eval_context(variables, hourly_df, config, job_ids, options_by_job):
    """Precompute arrays so the GA loop does not rebuild pandas objects thousands of times."""
    hours = np.array(variables["hours"], dtype=int)
    clusters = list(variables["clusters"])
    cluster_index = {cluster: idx for idx, cluster in enumerate(clusters)}
    hourly_lookup = hourly_df.set_index("hour")

    capacities = np.array(
        [
            [
                float(variables["cluster_data"][cluster]["capacity"]),
                float(variables["cluster_data"][cluster].get("gpu_capacity", 0.0)),
                float(variables["cluster_data"][cluster].get("cpu_capacity", 0.0)),
                float(variables["cluster_data"][cluster].get("memory_capacity_gb", 0.0)),
            ]
            for cluster in clusters
        ],
        dtype=float,
    )

    jobs = variables["jobs"]
    job_power = np.array([float(jobs[job_id]["power"]) for job_id in job_ids], dtype=float)
    job_duration = np.array([int(jobs[job_id]["duration"]) for job_id in job_ids], dtype=int)
    job_gpu = np.array([float(jobs[job_id].get("gpu_count_required", 0.0)) for job_id in job_ids], dtype=float)
    job_cpu = np.array([float(jobs[job_id].get("cpu_required", 0.0)) for job_id in job_ids], dtype=float)
    job_memory = np.array([float(jobs[job_id].get("memory_required_gb", 0.0)) for job_id in job_ids], dtype=float)

    option_cluster_indices = []
    option_starts = []
    for job_id in job_ids:
        clusters_for_job = []
        starts_for_job = []
        for cluster, start in options_by_job[job_id]:
            clusters_for_job.append(cluster_index[cluster])
            starts_for_job.append(int(start))
        option_cluster_indices.append(np.array(clusters_for_job, dtype=int))
        option_starts.append(np.array(starts_for_job, dtype=int))

    return {
        "hours": hours,
        "clusters": clusters,
        "cluster_index": cluster_index,
        "capacities": capacities,
        "baseline": np.array([float(variables["baseline_load"].get(int(hour), 0.0)) for hour in hours], dtype=float),
        "pue": np.array([float(hourly_lookup.loc[int(hour), "pue"]) if "pue" in hourly_lookup.columns else float(config.pue) for hour in hours], dtype=float),
        "renewable_available": np.array([float(hourly_lookup.loc[int(hour), "renewable_available"]) for hour in hours], dtype=float),
        "grid_price": np.array([float(hourly_lookup.loc[int(hour), "grid_price"]) for hour in hours], dtype=float),
        "job_power": job_power,
        "job_duration": job_duration,
        "job_gpu": job_gpu,
        "job_cpu": job_cpu,
        "job_memory": job_memory,
        "option_cluster_indices": option_cluster_indices,
        "option_starts": option_starts,
    }


def build_tables_from_loads(individual, loads, flexible_load, variables, hourly_df, config, job_ids, options_by_job, context):
    assignments = decode_individual(individual, job_ids, options_by_job)
    jobs = variables["jobs"]
    cluster_data = variables["cluster_data"]
    hourly_lookup = hourly_df.set_index("hour")

    schedule_rows = []
    for job_id in job_ids:
        cluster, start = assignments[job_id]
        job = jobs[job_id]
        schedule_rows.append(
            {
                "job_id": job_id,
                "workload_family": job["workload_family"],
                "assigned_cluster": cluster,
                "cluster_role": cluster_data[cluster].get("cluster_role", ""),
                "cluster_gpu_type": cluster_data[cluster].get("gpu_type", ""),
                "start_hour": int(start),
                "duration": int(job["duration"]),
                "power": float(job["power"]),
                "gpu_count_required": float(job.get("gpu_count_required", 0.0)),
                "cpu_required": float(job.get("cpu_required", 0.0)),
                "memory_required_gb": float(job.get("memory_required_gb", 0.0)),
            }
        )

    hourly_rows = []
    for hour_idx, hour in enumerate(context["hours"]):
        baseline = context["baseline"][hour_idx]
        it_load = flexible_load[hour_idx] + baseline
        total_load = context["pue"][hour_idx] * it_load
        renewable_available = context["renewable_available"][hour_idx]
        grid_price = context["grid_price"][hour_idx]
        renewable_consumption = min(renewable_available, total_load) if config.renewable_price <= grid_price else 0.0
        grid_consumption = total_load - renewable_consumption
        hourly_rows.append(
            {
                "hour": int(hour),
                "baseline_load": baseline,
                "pue": context["pue"][hour_idx],
                "flexible_load": flexible_load[hour_idx],
                "it_load": it_load,
                "total_load": total_load,
                "facility_load": total_load,
                "renewable_available": renewable_available,
                "renewable_consumption": renewable_consumption,
                "renewable_curtailment": max(0.0, renewable_available - renewable_consumption),
                "grid_consumption": grid_consumption,
                "grid_price": grid_price,
            }
        )

    return (
        pd.DataFrame(schedule_rows).sort_values(["start_hour", "job_id"]).reset_index(drop=True),
        pd.DataFrame(hourly_rows),
    )


def evaluate_individual(individual, variables, hourly_df, config, job_ids, options_by_job, penalty_weight, context, *, include_tables=False):
    n_clusters = len(context["clusters"])
    n_hours = len(context["hours"])
    loads = np.zeros((n_clusters, n_hours, 4), dtype=float)  # power, gpu, cpu, memory
    flexible_load = np.zeros(n_hours, dtype=float)

    for job_pos, gene in enumerate(individual):
        cluster_idx = context["option_cluster_indices"][job_pos][int(gene)]
        start = context["option_starts"][job_pos][int(gene)]
        duration = context["job_duration"][job_pos]
        active = (context["hours"] >= start) & (context["hours"] <= start + duration - 1)
        loads[cluster_idx, active, 0] += context["job_power"][job_pos]
        loads[cluster_idx, active, 1] += context["job_gpu"][job_pos]
        loads[cluster_idx, active, 2] += context["job_cpu"][job_pos]
        loads[cluster_idx, active, 3] += context["job_memory"][job_pos]
        flexible_load[active] += context["job_power"][job_pos]

    positive_capacities = context["capacities"] > 0
    excess = np.maximum(loads - context["capacities"][:, None, :], 0.0)
    normalized_excess = np.divide(
        excess,
        context["capacities"][:, None, :],
        out=np.zeros_like(excess),
        where=positive_capacities[:, None, :],
    )
    violation = float(normalized_excess.sum())

    total_load = context["pue"] * (flexible_load + context["baseline"])
    renewable_consumption = np.where(
        config.renewable_price <= context["grid_price"],
        np.minimum(context["renewable_available"], total_load),
        0.0,
    )
    grid_consumption = total_load - renewable_consumption
    renewable_cost = float((renewable_consumption * config.renewable_price * config.delta_t).sum())
    grid_cost = float((grid_consumption * context["grid_price"] * config.delta_t).sum())
    peak_over = max(0.0, float(grid_consumption.max()) - config.contracted_power)
    cost = renewable_cost + grid_cost + config.peak_price * peak_over
    fitness = cost + penalty_weight * violation
    feasible = violation <= 1e-9

    result = {"fitness": fitness, "cost": cost, "feasible": feasible, "violation": violation}
    if include_tables:
        resource_names = np.array(["power", "gpu", "cpu", "memory"])
        violation_rows = []
        for cluster_idx, hour_idx, resource_idx in np.argwhere(normalized_excess > 1e-9):
            violation_rows.append(
                {
                    "cluster_id": context["clusters"][int(cluster_idx)],
                    "hour": int(context["hours"][int(hour_idx)]),
                    "resource": str(resource_names[int(resource_idx)]),
                    "load": float(loads[cluster_idx, hour_idx, resource_idx]),
                    "capacity": float(context["capacities"][cluster_idx, resource_idx]),
                    "excess": float(excess[cluster_idx, hour_idx, resource_idx]),
                    "normalized_excess": float(normalized_excess[cluster_idx, hour_idx, resource_idx]),
                }
            )
        schedule_df, hourly_results = build_tables_from_loads(
            individual, loads, flexible_load, variables, hourly_df, config, job_ids, options_by_job, context
        )
        result.update(
            {
                "schedule": schedule_df,
                "hourly_results": hourly_results,
                "violations": pd.DataFrame(violation_rows),
                "metrics": compute_summary_metrics(hourly_results, config),
            }
        )
    return result


context = build_eval_context(variables, hourly_df, config, job_ids, options_by_job)
sample_individual = [0 for _ in job_ids]
sample_eval = evaluate_individual(
    sample_individual, variables, hourly_df, config, job_ids, options_by_job, GA_PENALTY_WEIGHT, context, include_tables=True
)
{key: sample_eval[key] for key in ["cost", "feasible", "violation", "fitness"]}

{'cost': 29.289710139166672,
 'feasible': True,
 'violation': 0.0,
 'fitness': 29.289710139166672}

In [5]:
def random_individual(rng, job_ids, options_by_job):
    return np.array([rng.integers(0, len(options_by_job[job_id])) for job_id in job_ids], dtype=int)


def repair_individual(individual, variables, hourly_df, config, job_ids, options_by_job, ga_config: GAConfig, rng, context):
    """Greedy repair: change one gene at a time when it reduces normalized capacity violation."""
    candidate = np.array(individual, dtype=int, copy=True)
    current = evaluate_individual(candidate, variables, hourly_df, config, job_ids, options_by_job, ga_config.penalty_weight, context)
    if current["feasible"]:
        return candidate, current

    for _ in range(ga_config.repair_passes):
        best_candidate = None
        best_eval = current
        job_order = rng.permutation(len(job_ids))
        for job_position in job_order:
            job_id = job_ids[int(job_position)]
            option_indices = list(range(len(options_by_job[job_id])))
            rng.shuffle(option_indices)
            for option_index in option_indices:
                if option_index == candidate[job_position]:
                    continue
                trial = candidate.copy()
                trial[job_position] = option_index
                trial_eval = evaluate_individual(
                    trial, variables, hourly_df, config, job_ids, options_by_job, ga_config.penalty_weight, context
                )
                improves_violation = trial_eval["violation"] < best_eval["violation"] - 1e-12
                same_violation_cheaper = abs(trial_eval["violation"] - best_eval["violation"]) <= 1e-12 and trial_eval["cost"] < best_eval["cost"]
                if improves_violation or same_violation_cheaper:
                    best_candidate = trial
                    best_eval = trial_eval
                if best_eval["feasible"]:
                    return best_candidate, best_eval
        if best_candidate is None:
            break
        candidate = best_candidate
        current = best_eval
    return candidate, current


def tournament_select(population, evaluations, ga_config: GAConfig, rng):
    contenders = rng.choice(len(population), size=ga_config.tournament_size, replace=False)
    best_index = min(contenders, key=lambda idx: evaluations[int(idx)]["fitness"])
    return population[int(best_index)].copy()


def crossover(parent_a, parent_b, rng):
    mask = rng.random(len(parent_a)) < 0.5
    child = parent_a.copy()
    child[mask] = parent_b[mask]
    return child


def mutate(individual, job_ids, options_by_job, mutation_rate, rng):
    child = individual.copy()
    for i, job_id in enumerate(job_ids):
        if rng.random() < mutation_rate and len(options_by_job[job_id]) > 1:
            current = child[i]
            new_value = rng.integers(0, len(options_by_job[job_id]) - 1)
            if new_value >= current:
                new_value += 1
            child[i] = new_value
    return child


def run_genetic_algorithm(variables, hourly_df, config, job_ids, options_by_job, ga_config: GAConfig, context):
    rng = np.random.default_rng(ga_config.random_seed)
    population = []
    evaluations = []

    for _ in range(ga_config.population_size):
        individual = random_individual(rng, job_ids, options_by_job)
        individual, evaluation = repair_individual(
            individual, variables, hourly_df, config, job_ids, options_by_job, ga_config, rng, context
        )
        population.append(individual)
        evaluations.append(evaluation)

    history = []
    best_feasible = None
    best_feasible_individual = None

    for generation in range(ga_config.generations + 1):
        best_eval = min(evaluations, key=lambda item: item["fitness"])
        feasible_indices = [idx for idx, item in enumerate(evaluations) if item["feasible"]]
        if feasible_indices:
            generation_best_index = min(feasible_indices, key=lambda idx: evaluations[idx]["cost"])
            generation_best_feasible = evaluations[generation_best_index]
            if best_feasible is None or generation_best_feasible["cost"] < best_feasible["cost"]:
                best_feasible = generation_best_feasible
                best_feasible_individual = population[generation_best_index].copy()

        history.append(
            {
                "generation": generation,
                "best_fitness": best_eval["fitness"],
                "best_cost": best_eval["cost"],
                "best_violation": best_eval["violation"],
                "feasible_count": len(feasible_indices),
                "best_feasible_cost": np.nan if best_feasible is None else best_feasible["cost"],
            }
        )
        if generation == ga_config.generations:
            break

        ranked_indices = sorted(range(len(population)), key=lambda idx: evaluations[idx]["fitness"])
        next_population = [population[idx].copy() for idx in ranked_indices[: ga_config.elite_count]]
        next_evaluations = [evaluations[idx] for idx in ranked_indices[: ga_config.elite_count]]

        while len(next_population) < ga_config.population_size:
            parent_a = tournament_select(population, evaluations, ga_config, rng)
            parent_b = tournament_select(population, evaluations, ga_config, rng)
            child = crossover(parent_a, parent_b, rng)
            child = mutate(child, job_ids, options_by_job, ga_config.mutation_rate, rng)
            child, child_eval = repair_individual(
                child, variables, hourly_df, config, job_ids, options_by_job, ga_config, rng, context
            )
            next_population.append(child)
            next_evaluations.append(child_eval)

        population = next_population
        evaluations = next_evaluations

    if best_feasible is None:
        best_index = min(range(len(population)), key=lambda idx: evaluations[idx]["fitness"])
        best_feasible_individual = population[best_index].copy()
        best_feasible = evaluations[best_index]

    final_eval = evaluate_individual(
        best_feasible_individual,
        variables,
        hourly_df,
        config,
        job_ids,
        options_by_job,
        ga_config.penalty_weight,
        context,
        include_tables=True,
    )
    return best_feasible_individual, final_eval, pd.DataFrame(history)


ga_config = GAConfig(
    population_size=GA_POPULATION_SIZE,
    generations=GA_GENERATIONS,
    mutation_rate=GA_MUTATION_RATE,
    tournament_size=GA_TOURNAMENT_SIZE,
    elite_count=GA_ELITE_COUNT,
    repair_passes=GA_REPAIR_PASSES,
    penalty_weight=GA_PENALTY_WEIGHT,
    random_seed=GA_RANDOM_SEED,
)

t0 = perf_counter()
ga_individual, ga_result, ga_history = run_genetic_algorithm(
    variables, hourly_df, config, job_ids, options_by_job, ga_config, context
)
ga_runtime = perf_counter() - t0

print(f"GA runtime: {ga_runtime:.2f}s")
print({key: ga_result[key] for key in ["cost", "feasible", "violation", "fitness"]})
ga_history.tail()

GA runtime: 0.67s
{'cost': 29.289710139166672, 'feasible': True, 'violation': 0.0, 'fitness': 29.289710139166672}


,generation,best_fitness,best_cost,best_violation,feasible_count,best_feasible_cost
76,76,29.28971,29.28971,0.0,60,29.28971
77,77,29.28971,29.28971,0.0,60,29.28971
78,78,29.28971,29.28971,0.0,60,29.28971
79,79,29.28971,29.28971,0.0,60,29.28971
80,80,29.28971,29.28971,0.0,60,29.28971


In [6]:
# Final GA feasibility report. This is the hard acceptance check.
if ga_result["feasible"]:
    print("GA produced a feasible schedule under all checked aggregate resource constraints.")
else:
    print("GA did not produce a feasible schedule. Largest violations:")

ga_result["violations"].sort_values("normalized_excess", ascending=False).head(10) if not ga_result["violations"].empty else ga_result["violations"]

GA produced a feasible schedule under all checked aggregate resource constraints.


""


In [7]:
ga_schedule = ga_result["schedule"]
ga_hourly = ga_result["hourly_results"]
ga_metrics = ga_result["metrics"]

print("GA metrics")
display(pd.Series(ga_metrics).to_frame("GA"))

ga_schedule.head(20)

GA metrics


,GA
renewable_cost,12.930097
grid_cost,16.359613
energy_cost,29.289710
peak_load,0.063040
peak_grid_import,0.010000
contracted_power,0.222000
peak_over_contracted,0.000000
peak_cost,0.000000
total_cost,29.289710
pue,1.000000


,job_id,workload_family,assigned_cluster,cluster_role,cluster_gpu_type,start_hour,duration,power,gpu_count_required,cpu_required,memory_required_gb
0,FT-08,fine_tuning,cluster_g3,alibaba_gpu_type_pool,G3,8,4,0.00350,4.0,32.0,192.0
1,FT-11,fine_tuning,cluster_g3,alibaba_gpu_type_pool,G3,8,4,0.00358,4.0,32.0,192.0
2,FT-01,fine_tuning,cluster_v100m32,alibaba_gpu_type_pool,V100M32,9,4,0.00354,4.0,32.0,192.0
3,FT-03,fine_tuning,cluster_v100m16,alibaba_gpu_type_pool,V100M16,9,4,0.00353,4.0,32.0,192.0
4,FT-04,fine_tuning,cluster_v100m32,alibaba_gpu_type_pool,V100M32,9,4,0.00354,4.0,32.0,192.0
5,FT-07,fine_tuning,cluster_v100m32,alibaba_gpu_type_pool,V100M32,9,4,0.00355,4.0,32.0,192.0
6,FT-10,fine_tuning,cluster_g3,alibaba_gpu_type_pool,G3,9,4,0.00350,4.0,32.0,192.0
7,FT-14,fine_tuning,cluster_g2,alibaba_gpu_type_pool,G2,9,4,0.00354,4.0,32.0,192.0
8,FT-02,fine_tuning,cluster_t4,alibaba_gpu_type_pool,T4,10,4,0.00359,4.0,32.0,192.0
9,FT-06,fine_tuning,cluster_v100m16,alibaba_gpu_type_pool,V100M16,10,4,0.00352,4.0,32.0,192.0


## MILP reference solve

The MILP is the reference implementation for this notebook. We solve the same job subset, same hourly inputs, same clusters, same PUE, and no battery. If MILP times out but has an incumbent solution, the helper still returns the incumbent; otherwise it raises.


In [8]:
milp_model, milp_variables = build_milp_model(jobs_df, hourly_df, clusters_df, config)
milp_model.Params.OutputFlag = 0

t0 = perf_counter()
try:
    solve_model(milp_model, time_limit=MILP_TIME_LIMIT_SECONDS)
    milp_runtime = perf_counter() - t0
    milp_schedule = extract_schedule(jobs_df, milp_variables)
    milp_hourly = extract_hourly_results(hourly_df, milp_variables)
    milp_metrics = compute_summary_metrics(milp_hourly, config)
    milp_status = milp_model.Status
    milp_gap = milp_model.MIPGap if milp_model.SolCount else np.nan
    print(f"MILP status: {milp_status}; runtime: {milp_runtime:.2f}s; gap: {milp_gap:.6f}")
    display(pd.Series(milp_metrics).to_frame("MILP"))
except Exception as exc:
    milp_runtime = perf_counter() - t0
    milp_schedule = pd.DataFrame()
    milp_hourly = pd.DataFrame()
    milp_metrics = None
    milp_status = getattr(milp_model, "Status", None)
    milp_gap = np.nan
    print(f"MILP failed after {milp_runtime:.2f}s: {exc}")

MILP status: 2; runtime: 0.00s; gap: 0.000000


,MILP
renewable_cost,12.930097
grid_cost,16.359613
energy_cost,29.289710
peak_load,0.056030
peak_grid_import,0.010000
contracted_power,0.222000
peak_over_contracted,0.000000
peak_cost,0.000000
total_cost,29.289710
pue,1.000000


In [9]:
comparison_rows = [
    {
        "solver": "GA",
        "feasible": bool(ga_result["feasible"]),
        "total_cost": ga_result["cost"],
        "runtime_seconds": ga_runtime,
        "peak_grid_import": ga_metrics["peak_grid_import"],
        "grid_energy_mwh": ga_metrics["grid_energy_mwh"],
        "renewable_energy_mwh": ga_metrics["renewable_energy_mwh"],
        "violation": ga_result["violation"],
    }
]
if milp_metrics is not None:
    comparison_rows.append(
        {
            "solver": "MILP",
            "feasible": True,
            "total_cost": milp_metrics["total_cost"],
            "runtime_seconds": milp_runtime,
            "peak_grid_import": milp_metrics["peak_grid_import"],
            "grid_energy_mwh": milp_metrics["grid_energy_mwh"],
            "renewable_energy_mwh": milp_metrics["renewable_energy_mwh"],
            "violation": 0.0,
        }
    )

comparison = pd.DataFrame(comparison_rows)
comparison

,solver,feasible,total_cost,runtime_seconds,peak_grid_import,grid_energy_mwh,renewable_energy_mwh,violation
0,GA,True,29.28971,0.669311,0.01,0.128908,0.323252,0.0
1,MILP,True,29.28971,0.000933,0.01,0.128908,0.323252,0.0


## How to interpret this first GA

This is a constructive baseline, not yet a production scheduler.

The important property is that feasibility is externally checked. The GA is allowed to explore infeasible assignments internally, but a run is only successful when the final `violations` table is empty. This gives us a clear pass/fail criterion for absurd-instance testing and later benchmarking.

Recommended next experiments:

1. Run the default subset and confirm GA and MILP both find feasible schedules.
2. Increase `MAX_JOBS` gradually: 20, 30, 40, then `None` for the full 50-job instance.
3. Repeat for `light`, `tense`, and `limit`.
4. Compare cost gap, runtime, and feasibility rate across random seeds.
5. Add battery dispatch after the discrete schedule is selected, ideally as a small LP decoder before encoding battery decisions directly in the GA.
